2025_04 【必須スキル】マッピング表作成

In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
print(os.getcwd())

/app/src


In [3]:
# DBからエクスポートした必須スキルデータを取得
df = pd.read_csv("/app/data/cleansing_skill.csv")

In [4]:
# データの確認
df.head()

,id,company_name,required_skills,required_skills_tmp
0,106,トランスコスモス株式会社,必要な資格・条件\n特になし\n<推奨される要件>\n■ネイティブレベルの日本語スキル(日本...,必要な資格・条件\n特になし\n<推奨される要件>\n■ネイティブレベルの日本語スキル(日本...
1,5,株式会社クリーク・アンド・リバー社,求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...,求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...
2,136,リクルーティング・パートナーズ株式会社 人材紹介事業部,NaN,NaN
3,115,株式会社クリーク・アンド・リバー社,求める人材: \n【求めるスキル・経験】※必須\n・SQLを使ったデータ抽出・集計の経験\n...,求める人材: \n【求めるスキル・経験】※必須\n・SQLを使ったデータ抽出・集計の経験\n...
4,53,ハロー・テクノ株式会社,求めている人材\n◆Webシステムの実務経験がある方\n◆AWSやPHP、C言語などを使用し...,求めている人材\n◆Webシステムの実務経験がある方\n◆AWSやPHP、C言語などを使用し...


In [5]:
# required_skills_tmpのデータを改行を基準に分割する
required_skills_tmp_split = df['required_skills_tmp'].dropna().str.split('\n').explode()

In [6]:
# 分割したデータの確認
required_skills_tmp_split.head(10)

0                              必要な資格・条件
0                                  特になし
0                             <推奨される要件>
0         ■ネイティブレベルの日本語スキル(日本語能力試験N1相当)
0    ■ビジネスレベルの英語スキル *面接は日本語・英語で実施いたします。
0           ■エクセルを使用したデータ作成・分析スキル\n<歓迎>
0                           ■BIツールの使用経験
0                 ■コンタクトセンター/サポートセンター経験
0        ■海外渡航の経験がある方(留学や旅行、ワーキングホリデー等)
0                    求めている人材の情報量は適切ですか？
Name: required_skills_tmp, dtype: object

In [7]:
# 分割後の行数の確認
print(len(required_skills_tmp_split))

2251


In [8]:
# 出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
十分                                    157
不足                                    157
求めている人材の情報量は適切ですか？                    157
求める人材:                                112
＜必須条件＞                                 26
                                     ... 
・Oracle EBS会計またはSCM領域の経験（両方あると尚可）       1
・シニアアナリストまたは導入コンサルタント経験 5年以上            1
・事業開発やPMI経験                             1
・人材紹介業・派遣業での経験                          1
・ロジカルにコミュニケーションしながら周りを巻き込む力\n✅歓迎条件      1
Name: count, Length: 1175, dtype: int64


In [9]:
# カンマなどで区切って、単語が複数含まれていそうなので更に分割する
required_skills_tmp_split = required_skills_tmp_split.str.split(r'[\t ,、・：]').explode()
required_skills_tmp_split = required_skills_tmp_split.str.split()

print(required_skills_tmp_split.head(10))

0                            [必要な資格]
0                               [条件]
0                             [特になし]
0                        [<推奨される要件>]
0    [■ネイティブレベルの日本語スキル(日本語能力試験N1相当)]
0                   [■ビジネスレベルの英語スキル]
0                          [*面接は日本語]
0                      [英語で実施いたします。]
0                  [■エクセルを使用したデータ作成]
0                      [分析スキル\n<歓迎>]
Name: required_skills_tmp, dtype: object


In [10]:
# 分割後の行数の確認
print(len(required_skills_tmp_split))

4885


In [11]:
# 出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
[]                      1039
[十分]                     157
[求めている人材の情報量は適切ですか？]     157
[不足]                     157
[求める人材:]                 112
                        ... 
[分析スキル\n<歓迎>]              1
[■エクセルを使用したデータ作成]          1
[英語で実施いたします。]              1
[*面接は日本語]                  1
[■ビジネスレベルの英語スキル]           1
Name: count, Length: 1963, dtype: int64


In [12]:
# リスト形式を文字列に変換
required_skills_tmp_split = required_skills_tmp_split.apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
)

# 前後の空白を削除
required_skills_tmp_split = required_skills_tmp_split.str.strip()

In [13]:
# 接続詞や助詞での分割条件
split_pattern = r'[、,・：\t　とやまた]'

# 接続詞や助詞で分割
required_skills_tmp_split = required_skills_tmp_split.str.split(split_pattern).explode()

# 空白行やnanの削除
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != '']
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != 'nan']

print(required_skills_tmp_split.head(10))

0                            必要な資格
0                               条件
0                             特になし
0                        <推奨される要件>
0    ■ネイティブレベルの日本語スキル(日本語能力試験N1相当)
0                   ■ビジネスレベルの英語スキル
0                          *面接は日本語
0                           英語で実施い
0                                し
0                               す。
Name: required_skills_tmp, dtype: object


In [14]:
# 不要なワードのリストを作成
stopwords = [
    '必要な資格', '条件', '特になし', '不足', '十分', '求める人材:', '<推奨される要件>', '求めている人材','資格','経験','求め','適切',
    '求めている人材の情報量は適切ですか？', '＜必須条件＞', 'の', 'と', 'ため', 'など', 
    'や', 'に', 'が', 'を', 'は', 'で', 'から', 'する', 'ある', 'いる', 
    '■', '*', '・', '：', '<', '>', '\\n', '\\t', '', ' ','＋','【','】','？'
]

In [15]:
# stopwordに含まれているワードを削除
required_skills_tmp_split = required_skills_tmp_split[~required_skills_tmp_split.isin(stopwords)]

# 空白行とnanを削除
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != '']
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != 'nan']

In [16]:
# stopwords の内容を確認
print("Stopwords:", stopwords)

# required_skills_tmp_split のサンプルデータを確認
print("Sample data from required_skills_tmp_split:", required_skills_tmp_split.head(10).tolist())

Stopwords: ['必要な資格', '条件', '特になし', '不足', '十分', '求める人材:', '<推奨される要件>', '求めている人材', '資格', '経験', '求め', '適切', '求めている人材の情報量は適切ですか？', '＜必須条件＞', 'の', 'と', 'ため', 'など', 'や', 'に', 'が', 'を', 'は', 'で', 'から', 'する', 'ある', 'いる', '■', '*', '・', '：', '<', '>', '\\n', '\\t', '', ' ', '＋', '【', '】', '？']
Sample data from required_skills_tmp_split: ['■ネイティブレベルの日本語スキル(日本語能力試験N1相当)', '■ビジネスレベルの英語スキル', '*面接は日本語', '英語で実施い', 'し', 'す。', '■エクセルを使用し', 'データ作成', '分析スキル\\n<歓迎>', '■BIツールの使用経験']


In [17]:
# ワードの出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
い方                               73
Python                           22
Ruby                             17
経験】                              16
す。                               16
                                 ..
■ネイティブレベルの日本語スキル(日本語能力試験N1相当)     1
ダッシュボードの要件定義                      1
はマーケティングプロジェクトのリード経験              1
■コンタクトセンター/サポートセンター経験             1
■BIツールの使用経験                       1
Name: count, Length: 2443, dtype: int64


In [18]:
# 分割後の行数の確認
print(len(word_counts))

2443


In [19]:
# word_countsをデータフレームに変換
word_counts_df = word_counts.reset_index()
word_counts_df.columns = ['skill', 'count']

# csvとして保存
# word_counts_df.to_csv('/app/data/required_skills.csv', index=False)

In [21]:
# word_counts_df　アルファベットから始まる単語を抽出
alphabet_words = word_counts_df[word_counts_df['skill'].str.match(r'^[A-Za-z]')]

# word_counts_df アルファベット以外から始まる単語
non_alphabet_words = word_counts_df[~word_counts_df['skill'].str.match(r'^[A-Za-z]')]

In [23]:
# csvとして出力
alphabet_words.to_csv('/app/data/alphabet_words.csv', index=False, encoding='utf-8')

non_alphabet_words.to_csv('/app/data/non_alphabet_words.csv', index=False, encoding='utf-8')